In [ ]:
def main(datasources, start_date, end_date):
    """BigAlpha 2026 end-to-end submission, raw-target 15m v6.

    The model is trained from scratch on the fixed development interval and only
    uses original 15-minute quote/bar fields.  The injected test table is used
    exclusively for out-of-sample inference.

    Starting point: BigQuant's official public Transformer example.  This entry
    is an original implementation with a larger rule-compliant architecture,
    five-year training span, cross-sectionally normalized raw return labels, a
    metric-aligned daily pairwise ranking loss, a spatial encoder for the
    five-level raw order book, streaming normalization, chunked
    training/inference, strict coverage, and deterministic ranking.
    """
    import gc
    import random
    import time

    import dai
    import numpy as np
    import pandas as pd
    import structlog
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    logger = structlog.get_logger()

    # Fixed training data and hyperparameters. The platform-supplied dates are
    # deliberately never used to select training rows.
    TRAIN_TABLE = "bigalpha_2026_stock_bar15m"
    TARGET_TABLE = "bigalpha_2026_exposure"
    UNIVERSE_TABLE = "bigalpha_2026_instruments"
    TRAIN_START = "2020-01-01 00:00:00"
    TRAIN_END = "2024-12-31 23:59:59"
    SEED = 20260801
    SEQ_LEN = 128
    MAX_TRAIN_INSTRUMENTS = 480
    TRAIN_CHUNK_SIZE = 16
    INFER_CHUNK_SIZE = 64
    EPOCHS = 2
    BATCH_SIZE = 512
    TRAIN_DAYS_PER_STEP = 32
    LEARNING_RATE = 3.0e-4
    WEIGHT_DECAY = 1.0e-4

    # 39 original source fields (< 100).  date/instrument are identifiers only;
    # ret is used only as the future training target and never enters the model.
    FEATURE_COLS = [
        "adjust_factor", "pre_close", "open", "high", "low", "close",
        "deal_number", "volume", "amount",
        "ask_price1", "ask_price2", "ask_price3", "ask_price4", "ask_price5",
        "bid_price1", "bid_price2", "bid_price3", "bid_price4", "bid_price5",
        "ask_volume1", "ask_volume2", "ask_volume3", "ask_volume4", "ask_volume5",
        "bid_volume1", "bid_volume2", "bid_volume3", "bid_volume4", "bid_volume5",
        "ask_num_orders1", "ask_num_orders2", "ask_num_orders3", "ask_num_orders4", "ask_num_orders5",
        "bid_num_orders1", "bid_num_orders2", "bid_num_orders3", "bid_num_orders4", "bid_num_orders5",
    ]
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _pick_infer_table(source_map):
        if "bar15m" in source_map:
            return source_map["bar15m"]
        for key, value in source_map.items():
            if "15m" in str(key).lower() or "15m" in str(value).lower():
                return value
        raise KeyError("datasources must contain the competition 15-minute bar table")

    infer_table = _pick_infer_table(datasources)

    class StockTransformer(nn.Module):
        def __init__(self):
            super().__init__()
            d_model, nhead, n_layers, dim_ff = 96, 4, 3, 256
            # The first nine columns are bar-level scalars. The remaining 30
            # columns are six raw book channels (ask/bid price, volume, and
            # order count) across five adjacent levels. Small convolutions learn
            # the level geometry inside the model without handcrafted features.
            self.bar_encoder = nn.Sequential(
                nn.Linear(9, 32),
                nn.GELU(),
                nn.LayerNorm(32),
            )
            self.book_encoder = nn.Sequential(
                nn.Conv1d(6, 24, kernel_size=3, padding=1),
                nn.GELU(),
                nn.Conv1d(24, 32, kernel_size=3, padding=1),
                nn.GELU(),
                nn.AdaptiveAvgPool1d(1),
            )
            self.fusion = nn.Sequential(
                nn.Linear(64, d_model),
                nn.GELU(),
                nn.LayerNorm(d_model),
            )
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
            self.position = nn.Parameter(torch.zeros(1, SEQ_LEN + 1, d_model))
            layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_ff,
                dropout=0.10,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
            self.head = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, 48),
                nn.GELU(),
                nn.Dropout(0.05),
                nn.Linear(48, 1),
            )
            nn.init.normal_(self.cls_token, std=0.02)
            nn.init.normal_(self.position, std=0.02)

        def forward(self, x):
            batch_size, sequence_length, _ = x.shape
            bar_embedding = self.bar_encoder(x[..., :9])
            book = x[..., 9:].reshape(batch_size * sequence_length, 6, 5)
            book_embedding = self.book_encoder(book).squeeze(-1)
            book_embedding = book_embedding.reshape(batch_size, sequence_length, 32)
            h = self.fusion(torch.cat((bar_embedding, book_embedding), dim=-1))
            cls = self.cls_token.expand(x.shape[0], -1, -1)
            h = torch.cat((cls, h), dim=1) + self.position
            h = self.encoder(h)
            return self.head(h[:, 0]).squeeze(-1)

    model = StockTransformer().to(device)
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if not 100_000 <= parameter_count <= 100_000_000:
        raise RuntimeError(f"trainable parameter count out of bounds: {parameter_count}")
    logger.info(
        "model_initialized",
        device=str(device),
        trainable_parameters=parameter_count,
        input_fields=len(FEATURE_COLS),
    )

    def _instrument_pool(sd, ed):
        frame = dai.query(
            f"SELECT DISTINCT instrument FROM {UNIVERSE_TABLE}",
            filters={"date": [sd, ed]},
        ).df()
        return sorted(frame["instrument"].dropna().astype(str).unique().tolist())

    train_instruments = _instrument_pool(TRAIN_START, TRAIN_END)
    rng = np.random.default_rng(SEED)
    rng.shuffle(train_instruments)
    train_instruments = train_instruments[:MAX_TRAIN_INSTRUMENTS]
    if not train_instruments:
        raise RuntimeError("the fixed training universe is empty")

    def _chunks(values, size):
        for offset in range(0, len(values), size):
            yield values[offset:offset + size]

    raw_sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {{table}} ORDER BY instrument, date"

    def _query_raw(table, sd, ed, instruments):
        frame = dai.query(
            raw_sql.format(table=table),
            filters={"date": [sd, ed], "instrument": instruments},
        ).df()
        if frame.empty:
            return frame
        frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
        frame["instrument"] = frame["instrument"].astype(str)
        return frame.dropna(subset=["date", "instrument"])

    def _signed_log_array(frame):
        values = frame[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
        values[~np.isfinite(values)] = np.nan
        return np.sign(values) * np.log1p(np.abs(values))

    # Streaming train-only moments keep peak memory bounded and ensure the same
    # per-field transform is reused for every train and test instrument.
    sums = np.zeros(len(FEATURE_COLS), dtype=np.float64)
    sum_squares = np.zeros(len(FEATURE_COLS), dtype=np.float64)
    counts = np.zeros(len(FEATURE_COLS), dtype=np.int64)
    stats_start = time.time()
    for instrument_chunk in _chunks(train_instruments, TRAIN_CHUNK_SIZE):
        raw = _query_raw(TRAIN_TABLE, TRAIN_START, TRAIN_END, instrument_chunk)
        if raw.empty:
            continue
        values = _signed_log_array(raw)
        valid = np.isfinite(values)
        safe = np.where(valid, values, 0.0)
        sums += safe.sum(axis=0)
        sum_squares += np.square(safe).sum(axis=0)
        counts += valid.sum(axis=0)
        del raw, values, valid, safe
        gc.collect()
    if np.any(counts == 0):
        missing = [FEATURE_COLS[i] for i in np.flatnonzero(counts == 0)]
        raise RuntimeError(f"training fields have no finite values: {missing}")
    means = sums / counts
    variances = np.maximum(sum_squares / counts - np.square(means), 1.0e-6)
    stds = np.sqrt(variances)
    logger.info("normalization_ready", seconds=round(time.time() - stats_start, 2))

    # v6 restores v1's label construction while preserving v5's 15-minute
    # architecture. ret at t is contemporaneous, so its normalized value is
    # shifted back one observation to label the raw sequence ending at t-1.
    target = dai.query(
        f"SELECT date, instrument, ret FROM {TARGET_TABLE}",
        filters={"date": [TRAIN_START, TRAIN_END]},
    ).df()
    target["date"] = pd.to_datetime(target["date"], errors="coerce").dt.normalize()
    target["instrument"] = target["instrument"].astype(str)
    target["ret"] = pd.to_numeric(target["ret"], errors="coerce")
    target = target.replace([np.inf, -np.inf], np.nan).dropna(subset=["date", "instrument", "ret"])
    def _robust_daily_target(series):
        lo, hi = series.quantile([0.01, 0.99])
        clipped = series.clip(lo, hi)
        scale = clipped.std(ddof=0)
        if not np.isfinite(scale) or scale < 1.0e-6:
            return clipped * 0.0
        return (clipped - clipped.mean()) / scale

    target["target"] = target.groupby("date", sort=False)["ret"].transform(
        _robust_daily_target
    )
    target = target.sort_values(["instrument", "date"])
    target["signal_date"] = target.groupby("instrument", sort=False)["date"].shift(1)
    target = target.dropna(subset=["signal_date", "target"])[
        ["signal_date", "instrument", "target"]
    ]
    target = target.rename(columns={"signal_date": "date"})

    def _standardize(frame):
        values = _signed_log_array(frame)
        missing = ~np.isfinite(values)
        if missing.any():
            values[missing] = np.take(means, np.nonzero(missing)[1])
        values = (values - means) / stds
        return np.clip(values, -12.0, 12.0).astype(np.float32)

    def _windows(frame, sd, ed, target_lookup=None):
        windows, labels, keys = [], [], []
        if frame.empty:
            return None, None, None
        sd_ts = pd.Timestamp(sd).normalize()
        ed_ts = pd.Timestamp(ed).normalize()
        frame = frame.sort_values(["instrument", "date"]).reset_index(drop=True)
        all_features = _standardize(frame)
        for instrument, positions in frame.groupby("instrument", sort=False).indices.items():
            positions = np.asarray(positions, dtype=np.int64)
            dates = frame.loc[positions, "date"].dt.normalize().to_numpy()
            daily_last = np.flatnonzero(np.r_[dates[1:] != dates[:-1], True])
            features = all_features[positions]
            for last_pos in daily_last:
                day = pd.Timestamp(dates[last_pos])
                if day < sd_ts or day > ed_ts:
                    continue
                if target_lookup is not None:
                    label = target_lookup.get((day, instrument))
                    if label is None or not np.isfinite(label):
                        continue
                start_pos = max(0, last_pos - SEQ_LEN + 1)
                window = features[start_pos:last_pos + 1]
                if len(window) < SEQ_LEN:
                    window = np.pad(window, ((SEQ_LEN - len(window), 0), (0, 0)))
                windows.append(window)
                keys.append((day, instrument))
                if target_lookup is not None:
                    labels.append(np.float32(label))
        if not windows:
            return None, None, None
        x = np.stack(windows).astype(np.float32, copy=False)
        y = np.asarray(labels, dtype=np.float32) if target_lookup is not None else None
        key_frame = pd.DataFrame(keys, columns=["date", "instrument"])
        return x, y, key_frame

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loss_fn = nn.SmoothL1Loss(beta=0.5)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
    model.train()
    train_start_time = time.time()
    for epoch in range(EPOCHS):
        epoch_instruments = list(train_instruments)
        random.Random(SEED + epoch).shuffle(epoch_instruments)
        losses = []
        sample_count = 0
        for chunk_number, instrument_chunk in enumerate(
            _chunks(epoch_instruments, TRAIN_CHUNK_SIZE)
        ):
            chunk_target = target[target["instrument"].isin(instrument_chunk)]
            target_lookup = {
                (pd.Timestamp(row.date), row.instrument): float(row.target)
                for row in chunk_target.itertuples(index=False)
            }
            raw = _query_raw(TRAIN_TABLE, TRAIN_START, TRAIN_END, instrument_chunk)
            x, y, keys = _windows(raw, TRAIN_START, TRAIN_END, target_lookup)
            del raw, chunk_target, target_lookup
            if x is None:
                continue

            # Preserve daily cross-sections inside each optimizer step. Each
            # instrument chunk contributes a small but genuine same-day panel;
            # several days are packed together to keep GPU utilization high.
            date_values = pd.to_datetime(keys["date"]).astype("int64").to_numpy()
            order = np.argsort(date_values, kind="stable")
            _, starts, group_sizes = np.unique(
                date_values[order], return_index=True, return_counts=True
            )
            day_groups = [
                order[start:start + size]
                for start, size in zip(starts.tolist(), group_sizes.tolist())
            ]
            random.Random(SEED + epoch * 10_000 + chunk_number).shuffle(day_groups)
            for day_offset in range(0, len(day_groups), TRAIN_DAYS_PER_STEP):
                step_groups = day_groups[day_offset:day_offset + TRAIN_DAYS_PER_STEP]
                batch_indices = np.concatenate(step_groups)
                step_sizes = [len(group) for group in step_groups]
                batch_x = torch.from_numpy(x[batch_indices]).to(device, non_blocking=True)
                batch_y = torch.from_numpy(y[batch_indices]).to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                    prediction = model(batch_x)
                    regression_loss = loss_fn(prediction, batch_y)
                    ranking_terms = []
                    cursor = 0
                    for group_size in step_sizes:
                        group_prediction = prediction[cursor:cursor + group_size]
                        group_target = batch_y[cursor:cursor + group_size]
                        cursor += group_size
                        if group_size < 2:
                            continue
                        target_difference = group_target[:, None] - group_target[None, :]
                        prediction_difference = (
                            group_prediction[:, None] - group_prediction[None, :]
                        )
                        valid_pairs = torch.triu(
                            torch.ones_like(target_difference, dtype=torch.bool),
                            diagonal=1,
                        ) & (target_difference.abs() > 1.0e-6)
                        if valid_pairs.any():
                            direction = torch.sign(target_difference[valid_pairs])
                            ranking_terms.append(
                                F.softplus(
                                    -direction * prediction_difference[valid_pairs]
                                ).mean()
                            )
                    ranking_loss = (
                        torch.stack(ranking_terms).mean()
                        if ranking_terms
                        else regression_loss * 0.0
                    )
                    loss = 0.65 * ranking_loss + 0.35 * regression_loss
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                losses.append(float(loss.detach().cpu()))
                sample_count += len(batch_x)
            del x, y, keys, day_groups
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
        if sample_count == 0:
            raise RuntimeError("no training windows were constructed")
        logger.info(
            "training_epoch_complete",
            epoch=epoch + 1,
            samples=sample_count,
            mean_loss=float(np.mean(losses)),
        )
    logger.info("training_complete", seconds=round(time.time() - train_start_time, 2))

    # Chunked inference: each instrument's history stays contiguous, while raw
    # test rows are released immediately after their scores are computed.
    requested_start = pd.Timestamp(start_date).normalize()
    requested_end = pd.Timestamp(end_date).normalize()
    buffer_start = (requested_start - pd.Timedelta(days=60)).strftime("%Y-%m-%d 00:00:00")
    infer_instruments = _instrument_pool(start_date, end_date)
    model.eval()
    prediction_frames = []
    with torch.no_grad():
        for instrument_chunk in _chunks(infer_instruments, INFER_CHUNK_SIZE):
            raw = _query_raw(infer_table, buffer_start, end_date, instrument_chunk)
            x, _, keys = _windows(raw, requested_start, requested_end, None)
            del raw
            if x is None:
                continue
            outputs = []
            for offset in range(0, len(x), BATCH_SIZE):
                batch_x = torch.from_numpy(x[offset:offset + BATCH_SIZE]).to(
                    device, non_blocking=True
                )
                with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                    outputs.append(model(batch_x).float().cpu().numpy())
            keys["score"] = np.concatenate(outputs).astype(np.float64)
            prediction_frames.append(keys)
            del x, keys, outputs
            gc.collect()

    if prediction_frames:
        predictions = pd.concat(prediction_frames, ignore_index=True)
    else:
        predictions = pd.DataFrame(columns=["date", "instrument", "score"])
    predictions = predictions.drop_duplicates(["date", "instrument"], keep="last")

    universe = dai.query(
        f"SELECT date, instrument FROM {UNIVERSE_TABLE}",
        filters={"date": [start_date, end_date]},
    ).df()
    universe["date"] = pd.to_datetime(universe["date"], errors="coerce").dt.normalize()
    universe["instrument"] = universe["instrument"].astype(str)
    universe = universe.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"])
    result = universe.merge(predictions, on=["date", "instrument"], how="left")
    result["score"] = pd.to_numeric(result["score"], errors="coerce")
    result["score"] = result["score"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    # Metrics are rank based; daily percentile ranks damp model-output outliers.
    result["score"] = result.groupby("date", sort=False)["score"].rank(
        method="average", pct=True
    ) - 0.5
    result = result.sort_values(["date", "instrument"])[["date", "instrument", "score"]]
    result = result.reset_index(drop=True)
    if result.empty or result["score"].isna().any():
        raise RuntimeError("submission output is empty or contains missing scores")
    logger.info(
        "prediction_complete",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )
    return result
